In [7]:
from dotenv import load_dotenv
import os
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

load_dotenv()

api_key = os.getenv("GIGA_KEY")

llm = GigaChat(
    credentials=api_key,
    model="GigaChat-2",
    verify_ssl_certs=False,
    scope="GIGACHAT_API_PERS",
    temperature=0.2,
    max_tokens=1000
)

response = llm.invoke("Привет! Как дела?")
print(response.content)

Привет! Всё отлично, готов общаться и помогать тебе с любыми вопросами. А ты как настроение держишь сегодня?


In [10]:
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.

Текст заявки: {text}

Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.

Количество человек:"""
)

chain = basic_prompt | llm | StrOutputParser()

In [11]:
test_texts = [
    "Ищу квартиру для семьи из четырех человек на длительный срок",
    "Нужна студия для проживания одного человека рядом с метро",
    "Требуется двухкомнатная квартира для молодой пары",
    "Снимем жилье для троих студентов на учебный год",
    "Семья с двумя детьми ищет просторную квартиру",
]

for text in test_texts:
    result = chain.invoke({"text": text})
    print(f"Текст: {text}")
    print(f"Результат: {result}")
    print("---")

Текст: Ищу квартиру для семьи из четырех человек на длительный срок
Результат: 4
---
Текст: Нужна студия для проживания одного человека рядом с метро
Результат: 1
---
Текст: Требуется двухкомнатная квартира для молодой пары
Результат: 2
---
Текст: Снимем жилье для троих студентов на учебный год
Результат: 3
---
Текст: Семья с двумя детьми ищет просторную квартиру
Результат: 4
---


In [12]:
import pandas as pd

data = {
    "text": [
        "Ищу квартиру для семьи из четырех человек на длительный срок",
        "Нужна студия для проживания одного человека рядом с метро",
        "Требуется двухкомнатная квартира для молодой пары",
        "Снимем жилье для троих студентов на учебный год",
        "Семья с двумя детьми ищет просторную квартиру",
        "Однокомнатная квартира для одного командировочного",
        "Ищем жилье для пятерых коллег на время проекта",
        "Молодая семья с новорожденным ребенком ищет квартиру",
        "Нужна квартира для троих — я и двое друзей",
        "Семья из шести человек переезжает, нужен большой дом",
        "Снимаю квартиру для себя на полгода",
        "Ищем двушку для двоих студентов",
        "Семья с тремя детьми ищет просторное жилье",
        "Нужна квартира для пары с собакой",
        "Четыре подруги ищут квартиру на лето"
    ],
    "amount": [4, 1, 2, 3, 4, 1, 5, 3, 3, 6, 1, 2, 5, 2, 4]
}

df = pd.DataFrame(data)
df.to_csv('rental_01.csv', sep=';', index=False, encoding='utf-8-sig')
print("Файл создан!")
df

Файл создан!


,text,amount
0,Ищу квартиру для семьи из четырех человек на д...,4
1,Нужна студия для проживания одного человека ря...,1
2,Требуется двухкомнатная квартира для молодой пары,2
3,Снимем жилье для троих студентов на учебный год,3
4,Семья с двумя детьми ищет просторную квартиру,4
5,Однокомнатная квартира для одного командировоч...,1
6,Ищем жилье для пятерых коллег на время проекта,5
7,Молодая семья с новорожденным ребенком ищет кв...,3
8,Нужна квартира для троих — я и двое друзей,3
9,"Семья из шести человек переезжает, нужен больш...",6


In [5]:
import pandas as pd

df = pd.read_csv('rental_01.csv', sep=';')
print(df.head())

                                                text  amount
0  Ищу квартиру для семьи из четырех человек на д...       4
1  Нужна студия для проживания одного человека ря...       1
2  Требуется двухкомнатная квартира для молодой пары       2
3    Снимем жилье для троих студентов на учебный год       3
4      Семья с двумя детьми ищет просторную квартиру       4


In [13]:
results = []

for _, row in df.iterrows():
    text = row["text"]
    try:
        result = chain.invoke({"text": text})
        results.append(result.strip())
    except Exception as e:
        results.append(f"ERROR: {e}")

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
print("Готово!")
df

Готово!


,text,amount,result
0,Ищу квартиру для семьи из четырех человек на д...,4,4
1,Нужна студия для проживания одного человека ря...,1,1
2,Требуется двухкомнатная квартира для молодой пары,2,2
3,Снимем жилье для троих студентов на учебный год,3,3
4,Семья с двумя детьми ищет просторную квартиру,4,4
5,Однокомнатная квартира для одного командировоч...,1,1
6,Ищем жилье для пятерых коллег на время проекта,5,5
7,Молодая семья с новорожденным ребенком ищет кв...,3,4
8,Нужна квартира для троих — я и двое друзей,3,3
9,"Семья из шести человек переезжает, нужен больш...",6,6


In [14]:
df["result_num"] = pd.to_numeric(df["result"], errors='coerce')

total = len(df)
correct = (df["result_num"] == df["amount"]).sum()
errors = total - correct
accuracy = correct / total

print(f"Всего заявок: {total}")
print(f"Верных ответов: {correct}")
print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")

# Где ошиблись
mistakes = df[df["result_num"] != df["amount"]][["text", "amount", "result_num"]]
if len(mistakes) > 0:
    print("\nОшибки модели:")
    print(mistakes.to_string())
else:
    print("\nВсе ответы верны!")

Всего заявок: 15
Верных ответов: 14
Ошибок: 1
Точность: 93.3%

Ошибки модели:
                                                   text  amount  result_num
7  Молодая семья с новорожденным ребенком ищет квартиру       3           4
